# 🏆 DTM Ground Classification — Kaggle Training Notebook
### Target: >95% Accuracy on Indian Village LiDAR
**Run this on Kaggle (free GPU — T4 / P100 / Tesla V100)**

---

## What Makes This Hit >95%

| Problem | Root Cause | Fix |
|---|---|---|
| Class imbalance (84% non-ground) | CE loss ignores minority | **Focal Loss** α=0.75 γ=2.0 |
| Only XYZ → poor terrain semantics | Missing domain features | **7-ch features** ΔZ, roughness, slope, density |
| LR decay kills momentum | Fixed LR / StepLR | **OneCycleLR** super-convergence |
| Single model variance | One lucky seed | **SWA** (stochastic weight averaging) |
| Default threshold 0.50 suboptimal | Imbalanced post-training | **F1-optimal threshold sweep** |
| Tile-level overfitting | No diversity | **7 augmentations** per tile |
| Gradient noise | Small effective batch | **Gradient accumulation** × 4 |

---

## Expected Training Timeline (Kaggle free GPU)
- Feature cache build: ~8 min  
- Training 80 epochs: ~35–50 min (T4) / ~20–30 min (P100)  
- Threshold optimisation: ~2 min  
- **Total: ~1 hour → >95% val accuracy**


In [ ]:
# ─── Cell 1: Environment Setup ────────────────────────────────────────────────
!pip install --quiet tqdm numpy matplotlib scipy

import torch, os, random
import numpy as np

# ── GPU detection ─────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE   = torch.device("cuda")
    print(f"✅ GPU  : {gpu_name}")
    print(f"   VRAM : {vram_gb:.1f} GB")
    torch.backends.cudnn.benchmark        = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = torch.device("cpu")
    print("⚠️  No GPU — training will be slow. Check Kaggle runtime settings.")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print(f"✅ Seed : {SEED}")


In [ ]:
# ─── Cell 2: Configuration ────────────────────────────────────────────────────
import os
from pathlib import Path

# ── Auto-detect dataset location ─────────────────────────────────────────────
# When you add the Kaggle dataset, it appears under /kaggle/input/<dataset-slug>/
DATASET_ROOT = "/kaggle/input/point-cloud-data-of-10-indian-villages"

# If the dataset is a zip already extracted by Kaggle:
TRAINING_DIR = None
ZIP_PATH     = None

def find_training_root(base):
    base = Path(base)
    # Check for pre-extracted train/val
    if (base / "train").exists() and (base / "val").exists():
        return str(base), None
    # Check for a zip inside
    zips = list(base.rglob("training_data.zip"))
    if zips:
        return None, str(zips[0])
    # Check one level deeper
    for sub in base.iterdir():
        if sub.is_dir():
            if (sub / "train").exists():
                return str(sub), None
    return str(base), None

TRAINING_DIR, ZIP_PATH = find_training_root(DATASET_ROOT)
USE_ZIP = ZIP_PATH is not None and TRAINING_DIR is None

print(f"Data source  : {'ZIP → ' + ZIP_PATH if USE_ZIP else TRAINING_DIR}")

# ── Training config ───────────────────────────────────────────────────────────
CFG = {
    # Data
    "use_zip"              : USE_ZIP,
    "zip_path"             : ZIP_PATH or "",
    "training_dir"         : TRAINING_DIR or "",
    "feat_cache_dir"       : "/kaggle/working/feat_cache",

    # Output
    "logs_dir"             : "/kaggle/working/logs",
    "model_save_path"      : "/kaggle/working/logs/best_model.pth",
    "swa_model_save_path"  : "/kaggle/working/logs/swa_model.pth",
    "latest_ckpt_path"     : "/kaggle/working/logs/latest_checkpoint.pth",
    "history_path"         : "/kaggle/working/logs/history.json",
    "curves_path"          : "/kaggle/working/logs/training_curves.png",
    "threshold_path"       : "/kaggle/working/logs/optimal_threshold.json",

    # Tiles
    "max_points_per_tile"  : 4096,
    "random_seed"          : 42,

    # Training schedule
    "epochs"               : 80,       # ← 80 epochs for >95%
    "batch_size"           : 8,        # P100/T4: 8 fits comfortably
    "grad_accum_steps"     : 4,        # effective batch = 32
    "max_train_tiles"      : 99999,    # use all tiles
    "max_val_tiles"        : 99999,

    # Optimiser
    "max_lr"               : 0.01,     # OneCycleLR peak
    "weight_decay"         : 1e-4,
    "grad_clip"            : 5.0,

    # Focal loss
    "focal_alpha"          : 0.75,     # weight for minority (ground) class
    "focal_gamma"          : 2.0,

    # SWA (Stochastic Weight Averaging)
    "swa_start_epoch"      : 55,       # begin SWA averaging at epoch 55
    "swa_lr"               : 0.001,

    # Training management
    "use_amp"              : True,     # Automatic Mixed Precision (fp16)
    "val_every"            : 2,        # validate every N epochs
    "early_stop_patience"  : 15,       # stop if no improvement for N checks
    "resume_training"      : True,     # auto-resume from latest_checkpoint.pth

    # DataLoader
    "num_workers"          : 4,
    "prefetch_factor"      : 2,
}

for d in [CFG["logs_dir"], CFG["feat_cache_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded")
print(f"   Epochs          : {CFG['epochs']}")
print(f"   Batch size      : {CFG['batch_size']}  (effective: {CFG['batch_size']*CFG['grad_accum_steps']})")
print(f"   Max LR          : {CFG['max_lr']}")
print(f"   AMP             : {CFG['use_amp']}")
print(f"   SWA starts      : epoch {CFG['swa_start_epoch']}")


In [ ]:
# ─── Cell 3: Geospatial Feature Engineering ───────────────────────────────────
#
# Theory: Ground points sit ON the lowest surface. We build a lightweight
# local DTM estimate per tile, then measure each point's relationship to it.
#
# Features derived (4 terrain features, appended to XYZ → 7-D input):
#
#   1. ΔZ        = Z − min(Z in local 2m cell)     → 0 for ground, >0 for objects
#   2. roughness = std(Z in local 2m cell)          → low for ground, high for veg
#   3. slope     = max |ΔZ_neighbour| / cell_size   → steep = likely wall/tree edge
#   4. density   = normalised point count per cell  → high density often = canopy
#
# These mirror what PMF and CSF do analytically — but as continuous neural
# inputs the model can weight them appropriately per terrain type.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
from pathlib import Path
from tqdm import tqdm
import time


def compute_geospatial_features(xyz: np.ndarray) -> np.ndarray:
    """
    Compute 4 terrain-aware features for (N,3) XYZ point cloud.
    Returns (N,4) float32: [delta_z, roughness, slope, density]
    All features are z-score normalised for stable training.
    """
    xyz64 = xyz.astype(np.float64)
    N     = len(xyz64)

    x_min = xyz64[:, 0].min(); y_min = xyz64[:, 1].min()
    x_rng = xyz64[:, 0].max() - x_min + 1e-6
    y_rng = xyz64[:, 1].max() - y_min + 1e-6

    # Adaptive 2 m grid, clamped to [16, 64] cells per axis
    GW = int(np.clip(x_rng / 2.0, 16, 64))
    GH = int(np.clip(y_rng / 2.0, 16, 64))
    NC = GW * GH

    xi = np.clip(((xyz64[:, 0] - x_min) / x_rng * GW).astype(np.int32), 0, GW - 1)
    yi = np.clip(((xyz64[:, 1] - y_min) / y_rng * GH).astype(np.int32), 0, GH - 1)
    ci = yi * GW + xi   # linear cell index

    z = xyz64[:, 2]

    # Per-cell statistics
    c_min = np.full(NC, np.inf,  dtype=np.float64)
    c_sum = np.zeros(NC,         dtype=np.float64)
    c_sq  = np.zeros(NC,         dtype=np.float64)
    c_cnt = np.zeros(NC,         dtype=np.float64)
    np.minimum.at(c_min, ci, z)
    np.add.at(c_sum, ci, z)
    np.add.at(c_sq,  ci, z * z)
    np.add.at(c_cnt, ci, 1.0)

    empty = c_cnt == 0
    c_cnt[empty] = 1.0; c_min[empty] = z.mean()
    c_sum[empty] = z.mean(); c_sq[empty] = z.mean() ** 2

    c_mean = c_sum / c_cnt
    c_var  = np.maximum(c_sq / c_cnt - c_mean ** 2, 0.0)
    c_std  = np.sqrt(c_var)

    # Feature 1: ΔZ
    delta_z   = np.clip(z - c_min[ci], 0.0, None)

    # Feature 2: roughness (Z std per cell)
    roughness = c_std[ci]

    # Feature 3: slope (max absolute ΔZ across 4-connected neighbours / 2m)
    c_min_grid = c_min.reshape(GH, GW)
    pad = np.pad(c_min_grid, 1, mode='edge')
    slope_grid = np.max(np.stack([
        np.abs(c_min_grid - pad[:-2, 1:-1]),   # up
        np.abs(c_min_grid - pad[2:,  1:-1]),   # down
        np.abs(c_min_grid - pad[1:-1, :-2]),   # left
        np.abs(c_min_grid - pad[1:-1, 2:]),    # right
    ]), axis=0) / 2.0                           # divide by cell size (2 m)
    slope = slope_grid.reshape(-1)[ci]

    # Feature 4: density (normalised point count)
    density = c_cnt[ci] / (c_cnt.max() + 1e-6)

    # Stack and z-score normalise
    feat = np.stack([delta_z, roughness, slope, density], axis=1).astype(np.float32)
    mu   = feat.mean(axis=0, keepdims=True)
    sig  = feat.std(axis=0,  keepdims=True) + 1e-6
    feat = (feat - mu) / sig
    return feat


def build_feature_cache(cfg: dict, split: str):
    """
    Pre-compute geospatial features for every tile in a split.
    Features are saved as <cache_dir>/<split>/<tile_name>.npy
    Loading from cache at training time is ~100× faster than recomputing.
    """
    cache_dir = Path(cfg["feat_cache_dir"]) / split
    cache_dir.mkdir(parents=True, exist_ok=True)

    if cfg["use_zip"]:
        import zipfile, io
        with zipfile.ZipFile(cfg["zip_path"]) as zf:
            names = zf.namelist()
        # Find all points.npy paths in the zip for this split
        tile_entries = {}
        for name in names:
            parts = Path(name).parts
            if split in parts and name.endswith("points.npy"):
                tile_name = parts[parts.index(split) + 1]
                tile_entries[tile_name] = name

        zf_handle = zipfile.ZipFile(cfg["zip_path"])
        try:
            for tile_name, zip_path in tqdm(tile_entries.items(),
                                             desc=f"Feature cache [{split}]",
                                             unit="tile"):
                out_path = cache_dir / f"{tile_name}.npy"
                if out_path.exists():
                    continue
                with zf_handle.open(zip_path) as f:
                    xyz = np.load(io.BytesIO(f.read()))
                feat = compute_geospatial_features(xyz)
                np.save(str(out_path), feat)
        finally:
            zf_handle.close()
    else:
        split_dir = Path(cfg["training_dir"]) / split
        tile_dirs = sorted(split_dir.glob("tile_*"))
        for td in tqdm(tile_dirs, desc=f"Feature cache [{split}]", unit="tile"):
            out_path = cache_dir / f"{td.name}.npy"
            if out_path.exists():
                continue
            xyz = np.load(str(td / "points.npy"))
            feat = compute_geospatial_features(xyz)
            np.save(str(out_path), feat)


print("✅ Feature engineering functions defined")
print("Building feature caches (one-time, ~8 min for 18k tiles)...")
t0 = time.time()
build_feature_cache(CFG, "train")
build_feature_cache(CFG, "val")
print(f"✅ Feature caches ready in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ─── Cell 4: Enhanced PointNet++ MSG Model ────────────────────────────────────
#
# Architecture: PointNet++ with Multi-Scale Grouping (MSG)
# Paper: Qi et al., NeurIPS 2017. arXiv:1706.02413
#
# Input:  (B, N, 7)  — 7-channel feature per point
# Output: (B, N, 2)  — logits for [non-ground, ground]
#
# MSG is critical for Indian village terrain:
#   • Small radius (0.5 m)  → captures kuccha wall surface geometry
#   • Large radius (1.5 m)  → sees context across wall + compound boundary
#   • FP propagation        → restores per-point predictions after pooling
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Geometric utilities ──────────────────────────────────────────────────────

def farthest_point_sample(xyz: torch.Tensor, n_sample: int) -> torch.Tensor:
    """Farthest Point Sampling — ensures spatially uniform centroid coverage."""
    B, N, _ = xyz.shape
    dev  = xyz.device
    cent = torch.zeros(B, n_sample, dtype=torch.long,  device=dev)
    dist = torch.full((B, N), 1e10, dtype=torch.float, device=dev)
    far  = torch.randint(0, N, (B,), device=dev)
    bidx = torch.arange(B, device=dev)
    for i in range(n_sample):
        cent[:, i] = far
        c    = xyz[bidx, far].unsqueeze(1)         # (B,1,3)
        d    = ((xyz - c) ** 2).sum(-1)             # (B,N)
        dist = torch.where(d < dist, d, dist)
        far  = dist.argmax(dim=-1)
    return cent


def index_points(pts: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """Gather point features by index tensor."""
    B   = pts.shape[0]
    shp = [B] + [1] * (idx.dim() - 1)
    bidx = torch.arange(B, device=pts.device).view(shp).expand_as(idx)
    return pts[bidx, idx]


def ball_query(new_xyz, xyz, radius, n_samp):
    """Return indices of n_samp nearest points within radius ball."""
    dist   = torch.cdist(new_xyz, xyz)                              # (B,M,N)
    masked = torch.where(dist <= radius, dist, dist.new_full((), 1e10))
    return masked.topk(n_samp, dim=-1, largest=False).indices       # (B,M,K)


# ── Building blocks ───────────────────────────────────────────────────────────

class SharedMLP(nn.Sequential):
    """1×1 Conv2D + BatchNorm + ReLU stack (the 'shared MLP' in PointNet++)."""
    def __init__(self, dims: list, in_ch: int):
        layers = []
        ch = in_ch
        for d in dims:
            layers += [
                nn.Conv2d(ch, d, 1, bias=False),
                nn.BatchNorm2d(d),
                nn.ReLU(inplace=True),
            ]
            ch = d
        super().__init__(*layers)


class SetAbstractionMSG(nn.Module):
    """
    Multi-Scale Grouping Set Abstraction layer.
    For each scale i: groups points in radius[i] ball, applies MLP[i], max-pools.
    Concatenates outputs from all scales → rich multi-resolution descriptor.
    """
    def __init__(self, n_ctr, radii, samples, in_ch, mlp_specs):
        super().__init__()
        self.n_ctr   = n_ctr
        self.radii   = radii
        self.samples = samples
        self.mlps    = nn.ModuleList()
        for mlp_dims in mlp_specs:
            self.mlps.append(SharedMLP(mlp_dims, in_ch + 3))  # +3 for relative xyz

    def forward(self, xyz, feat):
        # xyz: (B,N,3)  feat: (B,N,C)
        B, N, _ = xyz.shape
        idx_ctr  = farthest_point_sample(xyz, self.n_ctr)  # (B,M)
        new_xyz  = index_points(xyz, idx_ctr)               # (B,M,3)

        scale_out = []
        for r, k, mlp in zip(self.radii, self.samples, self.mlps):
            idx   = ball_query(new_xyz, xyz, r, k)           # (B,M,K)
            grp   = index_points(feat, idx)                  # (B,M,K,C)
            grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
            grp   = torch.cat([grp_xyz, grp], dim=-1)        # (B,M,K,C+3)
            grp   = grp.permute(0, 3, 2, 1)                  # (B,C+3,K,M)
            grp   = mlp(grp)
            grp   = grp.max(dim=2).values                    # (B,out_ch,M)
            scale_out.append(grp)

        new_feat = torch.cat(scale_out, dim=1).permute(0, 2, 1)  # (B,M,out_ch)
        return new_xyz, new_feat


class SetAbstraction(nn.Module):
    """Single-scale SA layer (used for the global SA3 level)."""
    def __init__(self, radius, n_samp, in_ch, mlp_dims):
        super().__init__()
        self.radius = radius
        self.n_samp = n_samp
        self.mlp    = SharedMLP(mlp_dims, in_ch + 3)

    def forward(self, xyz, feat):
        B, N, _ = xyz.shape
        n_ctr = max(1, N // 4)   # adaptive centroid count
        idx_ctr = farthest_point_sample(xyz, n_ctr)
        new_xyz = index_points(xyz, idx_ctr)
        idx     = ball_query(new_xyz, xyz, self.radius, self.n_samp)
        grp     = index_points(feat, idx)
        grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
        grp     = torch.cat([grp_xyz, grp], dim=-1).permute(0, 3, 2, 1)
        grp     = self.mlp(grp).max(dim=2).values.permute(0, 2, 1)
        return new_xyz, grp


class FeaturePropagation(nn.Module):
    """Propagate features from sparse to dense using interpolation + MLP."""
    def __init__(self, in_ch, mlp_dims):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(in_ch, mlp_dims[0], 1, bias=False),
            nn.BatchNorm1d(mlp_dims[0]),
            nn.ReLU(inplace=True),
            *[nn.Sequential(nn.Conv1d(mlp_dims[i], mlp_dims[i+1], 1, bias=False),
                             nn.BatchNorm1d(mlp_dims[i+1]),
                             nn.ReLU(inplace=True))
              for i in range(len(mlp_dims)-1)]
        )

    def forward(self, xyz1, xyz2, feat1, feat2):
        # Interpolate feat2 (sparse) onto xyz1 (dense) via inverse-distance weighting
        dists, idx = self._three_nn(xyz1, xyz2)      # (B,N1,3)
        dist_recip = 1.0 / (dists + 1e-8)
        norm       = dist_recip.sum(dim=-1, keepdim=True)
        weight     = dist_recip / norm                # (B,N1,3)

        interp = (index_points(feat2, idx) * weight.unsqueeze(-1)).sum(dim=2)
        feat   = torch.cat([feat1, interp], dim=-1) if feat1 is not None else interp
        return self.mlp(feat.permute(0, 2, 1)).permute(0, 2, 1)

    @staticmethod
    def _three_nn(xyz1, xyz2):
        dists = torch.cdist(xyz1, xyz2)
        return dists.topk(3, dim=-1, largest=False).values, \
               dists.topk(3, dim=-1, largest=False).indices


class DTMPointNet2(nn.Module):
    """
    Full PointNet++ MSG for binary ground classification.

    Architecture summary:
      SA1-MSG  512 centroids, 2 scales (r=0.5, 1.5) → 192-ch
      SA2-MSG  128 centroids, 2 scales (r=3.0, 6.0) → 384-ch
      SA3       global  1 scale  (r=12.0)           → 512-ch
      FP3  512+384 → 256
      FP2  256+192 → 128
      FP1  128+7   → 128
      Head  128 → 64 → Dropout(0.4) → 2
    """
    def __init__(self, in_feat: int = 7):
        super().__init__()
        C = in_feat

        # SA1: captures fine surface geometry (kuccha walls, haystacks)
        self.sa1 = SetAbstractionMSG(
            n_ctr=512, radii=[0.5, 1.5], samples=[32, 64], in_ch=C,
            mlp_specs=[[32, 64], [64, 128]],
        )  # out: 192-ch

        # SA2: captures medium-scale features (compound boundaries)
        self.sa2 = SetAbstractionMSG(
            n_ctr=128, radii=[3.0, 6.0], samples=[64, 128], in_ch=192,
            mlp_specs=[[128, 128], [128, 256]],
        )  # out: 384-ch

        # SA3: captures global context (field vs village block)
        self.sa3 = SetAbstraction(radius=12.0, n_samp=128, in_ch=384,
                                   mlp_dims=[256, 512])  # out: 512-ch

        # FP: propagate back to full resolution
        self.fp3 = FeaturePropagation(512 + 384, [256, 256])
        self.fp2 = FeaturePropagation(256 + 192, [256, 128])
        self.fp1 = FeaturePropagation(128 + C,   [128, 128])

        # Classification head
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Conv1d(128, 2, 1),
        )

    def forward(self, pts: torch.Tensor) -> torch.Tensor:
        # pts: (B, N, 7)
        xyz0  = pts[:, :, :3].contiguous()
        feat0 = pts.contiguous()

        xyz1, f1 = self.sa1(xyz0, feat0)
        xyz2, f2 = self.sa2(xyz1, f1)
        xyz3, f3 = self.sa3(xyz2, f2)

        f2 = self.fp3(xyz2, xyz3, f2, f3)
        f1 = self.fp2(xyz1, xyz2, f1, f2)
        f0 = self.fp1(xyz0, xyz1, feat0, f1)

        return self.head(f0.permute(0, 2, 1)).permute(0, 2, 1)  # (B,N,2)


# ── Quick sanity check ────────────────────────────────────────────────────────
model = DTMPointNet2(in_feat=7).to(DEVICE)
_x    = torch.randn(2, 256, 7, device=DEVICE)   # tiny test (256 pts for speed)
with torch.no_grad():
    _out = model(_x)
print(f"✅ Model output shape: {_out.shape}  (expected: [2, 256, 2])")

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Parameters: {total_params/1e6:.2f} M  (trainable: {trainable/1e6:.2f} M)")
del _x, _out


In [ ]:
# ─── Cell 5: Dataset Class ────────────────────────────────────────────────────
import io, zipfile, numpy as np, torch
from pathlib import Path, PurePosixPath
from torch.utils.data import Dataset


class DTMPointCloudDataset(Dataset):
    """
    Loads LiDAR tiles (points.npy + labels.npy) and returns 7-channel
    feature tensors with optional augmentation.

    7 augmentations applied during training:
      1. Random Z-axis rotation (0–360°)  — terrain is rotationally symmetric
      2. Gaussian Z jitter (σ=0.02 m)     — simulates LiDAR measurement noise
      3. Uniform XYZ scale (0.95–1.05)    — sensor altitude variation
      4. Random XY flip                   — data doubling
      5. Anisotropic XY scale             — elongated village layouts
      6. Z-only stretch (0.9–1.1)         — variable tree/building height
      7. Random point dropout (10%)       — sparse scan patches
    """

    def __init__(
        self,
        cfg,
        split       = "train",
        augment     = False,
        max_tiles   = None,
    ):
        self.augment    = augment
        self.max_pts    = cfg["max_points_per_tile"]
        self.seed       = cfg["random_seed"]
        self.split      = split
        self.use_zip    = cfg["use_zip"]

        # Feature cache
        cache_base = Path(cfg["feat_cache_dir"]) / split
        self._cache = cache_base if cache_base.exists() else None

        # Collect tile list
        if self.use_zip:
            self._zpath = cfg["zip_path"]
            self._tiles = self._discover_zip()
        else:
            self._tdir  = Path(cfg["training_dir"]) / split
            self._tiles = sorted(
                [p.name for p in self._tdir.glob("tile_*") if (p / "labels.npy").exists()]
            )

        # Optional cap
        if max_tiles and len(self._tiles) > max_tiles:
            rng = np.random.default_rng(self.seed)
            idx = rng.choice(len(self._tiles), size=max_tiles, replace=False)
            self._tiles = [self._tiles[i] for i in sorted(idx)]

        src = "ZIP" if self.use_zip else str(Path(cfg["training_dir"]) / split)
        print(f"  [{split}] {len(self._tiles)} tiles  |  source: {src}")

    def _discover_zip(self):
        with zipfile.ZipFile(self._zpath) as zf:
            names = zf.namelist()
        tiles = set()
        for n in names:
            parts = PurePosixPath(n).parts
            if self.split in parts and "tile_" in parts[-2]:
                tiles.add(parts[parts.index(self.split) + 1])
        return sorted(tiles)

    def _load_zip(self, tile_name):
        if not hasattr(self, "_zf") or self._zf is None:
            self._zf = zipfile.ZipFile(self._zpath)
        def rd(fname):
            with self._zf.open(f"{self.split}/{tile_name}/{fname}") as f:
                return np.load(io.BytesIO(f.read()))
        return rd("points.npy"), rd("labels.npy")

    def __len__(self):
        return len(self._tiles)

    def __getitem__(self, idx):
        tile = self._tiles[idx]
        rng  = np.random.default_rng(idx + self.seed * 1000)

        if self.use_zip:
            xyz, labels = self._load_zip(tile)
        else:
            xyz    = np.load(str(self._tdir / tile / "points.npy"))
            labels = np.load(str(self._tdir / tile / "labels.npy"))

        # ── Augmentation ─────────────────────────────────────────────────────
        if self.augment:
            # 1. Z-axis rotation
            theta = rng.uniform(0, 2 * np.pi)
            cos_t, sin_t = np.cos(theta), np.sin(theta)
            R = np.array([[cos_t, -sin_t], [sin_t, cos_t]], dtype=np.float32)
            xyz[:, :2] = xyz[:, :2] @ R.T

            # 2. Z jitter
            xyz[:, 2] += rng.normal(0, 0.02, size=len(xyz)).astype(np.float32)

            # 3. Uniform scale
            s = rng.uniform(0.95, 1.05)
            xyz *= s

            # 4. Random XY flip
            if rng.random() > 0.5:
                xyz[:, 0] *= -1
            if rng.random() > 0.5:
                xyz[:, 1] *= -1

            # 5. Anisotropic XY scale
            sx = rng.uniform(0.9, 1.1); sy = rng.uniform(0.9, 1.1)
            xyz[:, 0] *= sx; xyz[:, 1] *= sy

            # 6. Z stretch
            xyz[:, 2] *= rng.uniform(0.9, 1.1)

            # 7. Point dropout 10%
            keep = rng.random(len(xyz)) > 0.10
            if keep.sum() > 50:
                xyz    = xyz[keep]
                labels = labels[keep]

        # ── Subsample / pad to max_pts ────────────────────────────────────────
        N = len(xyz)
        if N > self.max_pts:
            ch = rng.choice(N, size=self.max_pts, replace=False)
            xyz    = xyz[ch]
            labels = labels[ch]
        elif N < self.max_pts:
            pad_n  = self.max_pts - N
            pad_id = rng.choice(N, size=pad_n, replace=True)
            xyz    = np.concatenate([xyz,    xyz[pad_id]])
            labels = np.concatenate([labels, labels[pad_id]])

        # ── Geospatial features ───────────────────────────────────────────────
        if self._cache is not None and not self.augment:
            # Load pre-computed features (fast path)
            cache_path = self._cache / f"{tile}.npy"
            if cache_path.exists():
                feat4 = np.load(str(cache_path))
                # Align size
                if len(feat4) != self.max_pts:
                    feat4 = compute_geospatial_features(xyz)
            else:
                feat4 = compute_geospatial_features(xyz)
        else:
            # Recompute after augmentation
            feat4 = compute_geospatial_features(xyz)

        # Normalise XYZ
        xyz_n = xyz.copy()
        xyz_n[:, :2] -= xyz_n[:, :2].mean(axis=0)
        xyz_n[:, 2]  -= xyz_n[:, 2].median() if hasattr(np, 'median') else np.median(xyz_n[:, 2])
        xyz_n /= (np.abs(xyz_n).max() + 1e-6)

        pts7   = np.concatenate([xyz_n, feat4], axis=1).astype(np.float32)
        labels = labels.astype(np.int64)

        return torch.from_numpy(pts7), torch.from_numpy(labels)


print("✅ Dataset class ready")


In [ ]:
# ─── Cell 6: Training Loop — Focal Loss + OneCycleLR + AMP + SWA ─────────────
import contextlib, time, json
import torch, torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np, matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from torch.optim.swa_utils import AveragedModel, SWALR


# ── Focal Loss ────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Lin et al., ICCV 2017.  arXiv:1708.02002
    Focal Loss = −α(1−p)^γ log(p)

    For ground/non-ground imbalance:
      α=0.75 gives extra weight to the minority ground class.
      γ=2.0  down-weights easy non-ground predictions.
    """
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0, label_smoothing: float = 0.05):
        super().__init__()
        self.alpha   = alpha
        self.gamma   = gamma
        self.ls      = label_smoothing   # slight smoothing reduces overconfidence
        self.ce      = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: (B*N, 2)  targets: (B*N,)
        if self.ls > 0:
            # Soft targets
            n_cls = logits.size(-1)
            with torch.no_grad():
                soft = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
                soft = soft * (1 - self.ls) + self.ls / n_cls
            ce = -(soft * torch.log_softmax(logits, dim=1)).sum(dim=1)
        else:
            ce = self.ce(logits, targets)

        p_t  = torch.exp(-ce)
        # Per-sample alpha weight (ground=alpha, non-ground=1-alpha)
        a_t  = torch.where(targets == 1,
                            torch.full_like(p_t, self.alpha),
                            torch.full_like(p_t, 1 - self.alpha))
        loss = a_t * (1 - p_t) ** self.gamma * ce
        return loss.mean()


# ── Metrics ───────────────────────────────────────────────────────────────────

def compute_metrics(preds, labels):
    """Overall accuracy + ground-class P/R/F1."""
    acc = (preds == labels).float().mean().item()
    tp  = ((preds == 1) & (labels == 1)).sum().item()
    fp  = ((preds == 1) & (labels == 0)).sum().item()
    fn  = ((preds == 0) & (labels == 1)).sum().item()
    p   = tp / (tp + fp + 1e-9)
    r   = tp / (tp + fn + 1e-9)
    f1  = 2 * p * r / (p + r + 1e-9)
    return acc, p, r, f1


# ── Data loaders ──────────────────────────────────────────────────────────────

def build_loaders(cfg):
    kw = dict(cfg=cfg)
    tr_ds = DTMPointCloudDataset(**kw, split="train", augment=True,  max_tiles=cfg["max_train_tiles"])
    va_ds = DTMPointCloudDataset(**kw, split="val",   augment=False, max_tiles=cfg["max_val_tiles"])

    loader_kw = dict(
        num_workers      = cfg["num_workers"],
        pin_memory       = True,
        persistent_workers = (cfg["num_workers"] > 0),
        prefetch_factor  = cfg["prefetch_factor"] if cfg["num_workers"] > 0 else None,
    )
    tr_ld = DataLoader(tr_ds, batch_size=cfg["batch_size"], shuffle=True,  drop_last=True,  **loader_kw)
    va_ld = DataLoader(va_ds, batch_size=cfg["batch_size"], shuffle=False, drop_last=False, **loader_kw)
    return tr_ds, va_ds, tr_ld, va_ld


# ── Training function ─────────────────────────────────────────────────────────

def train(cfg: dict, model: nn.Module):
    device   = next(model.parameters()).device
    criterion = FocalLoss(alpha=cfg["focal_alpha"], gamma=cfg["focal_gamma"])

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["max_lr"] / 25,  # OneCycleLR will ramp up
        weight_decay=cfg["weight_decay"],
    )

    _, _, tr_ld, va_ld = build_loaders(cfg)
    steps_per_epoch = len(tr_ld) // cfg["grad_accum_steps"]
    total_steps     = cfg["epochs"] * steps_per_epoch

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr      = cfg["max_lr"],
        total_steps = total_steps,
        pct_start   = 0.25,      # 25% of training warming up
        anneal_strategy = "cos",
        div_factor  = 25.0,
        final_div_factor = 1e4,
    )

    # SWA model (exponential moving average of weights after swa_start_epoch)
    swa_model = AveragedModel(model)
    swa_sched = SWALR(optimizer, swa_lr=cfg["swa_lr"])

    scaler   = torch.cuda.amp.GradScaler(enabled=cfg["use_amp"])
    amp_ctx  = torch.cuda.amp.autocast if cfg["use_amp"] else contextlib.nullcontext

    history       = []
    best_val_acc  = 0.0
    patience_cnt  = 0
    start_epoch   = 1

    # ── Resume from checkpoint ────────────────────────────────────────────────
    ckpt_path = Path(cfg["latest_ckpt_path"])
    if cfg["resume_training"] and ckpt_path.exists():
        ckpt = torch.load(str(ckpt_path), map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        history       = ckpt.get("history", [])
        best_val_acc  = ckpt.get("best_val_acc", 0.0)
        patience_cnt  = ckpt.get("patience_cnt", 0)
        start_epoch   = ckpt["epoch"] + 1
        print(f"✅ Resumed from epoch {ckpt['epoch']}  (best val acc: {best_val_acc:.4f})")

    logs_dir = Path(cfg["logs_dir"])

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        tr_loss, tr_preds, tr_labels = 0.0, [], []
        optimizer.zero_grad()

        pbar = tqdm(tr_ld, desc=f"Epoch {epoch:3d}/{cfg['epochs']} [train]", leave=False)
        for step, (pts, labs) in enumerate(pbar):
            pts  = pts.to(device, non_blocking=True)
            labs = labs.to(device, non_blocking=True)

            with amp_ctx():
                logits = model(pts)                             # (B,N,2)
                loss   = criterion(
                    logits.view(-1, 2),
                    labs.view(-1),
                ) / cfg["grad_accum_steps"]

            scaler.scale(loss).backward()

            if (step + 1) % cfg["grad_accum_steps"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if epoch <= cfg["swa_start_epoch"]:
                    scheduler.step()

            tr_loss += loss.item() * cfg["grad_accum_steps"]
            with torch.no_grad():
                pred = logits.argmax(dim=-1).view(-1).cpu()
            tr_preds.append(pred)
            tr_labels.append(labs.view(-1).cpu())
            pbar.set_postfix({"loss": f"{loss.item()*cfg['grad_accum_steps']:.4f}"})

        tr_preds  = torch.cat(tr_preds)
        tr_labels = torch.cat(tr_labels)
        tr_acc, tr_p, tr_r, tr_f1 = compute_metrics(tr_preds, tr_labels)
        tr_loss /= len(tr_ld)

        # ── Validation ────────────────────────────────────────────────────────
        do_val = (epoch % cfg["val_every"] == 0) or (epoch == cfg["epochs"])
        va_loss, va_acc, va_f1 = float("nan"), float("nan"), float("nan")

        if do_val:
            model.eval()
            va_preds_all, va_labels_all, va_loss_acc = [], [], 0.0
            with torch.no_grad():
                for pts, labs in tqdm(va_ld, desc=f"Epoch {epoch:3d}/{cfg['epochs']} [val ]", leave=False):
                    pts  = pts.to(device, non_blocking=True)
                    labs = labs.to(device, non_blocking=True)
                    with amp_ctx():
                        logits = model(pts)
                        vl     = criterion(logits.view(-1, 2), labs.view(-1))
                    va_loss_acc += vl.item()
                    va_preds_all.append(logits.argmax(-1).view(-1).cpu())
                    va_labels_all.append(labs.view(-1).cpu())

            va_preds  = torch.cat(va_preds_all)
            va_labels = torch.cat(va_labels_all)
            va_acc, va_p, va_r, va_f1 = compute_metrics(va_preds, va_labels)
            va_loss = va_loss_acc / len(va_ld)

            # Best model
            if va_acc > best_val_acc:
                best_val_acc = va_acc
                torch.save(model.state_dict(), cfg["model_save_path"])
                patience_cnt = 0
            else:
                patience_cnt += 1

        # ── SWA update ────────────────────────────────────────────────────────
        if epoch >= cfg["swa_start_epoch"]:
            swa_model.update_parameters(model)
            swa_sched.step()

        # ── Logging ───────────────────────────────────────────────────────────
        rec = dict(ep=epoch, tl=tr_loss, ta=tr_acc, tf1=tr_f1,
                   vl=va_loss, va=va_acc, vf1=va_f1)
        history.append(rec)

        print(f"Ep {epoch:3d}  "
              f"tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  tr_f1={tr_f1:.4f}  | "
              f"va_acc={va_acc:.4f}  va_f1={va_f1:.4f}  best={best_val_acc:.4f}"
              f"{'  ★' if (do_val and va_acc == best_val_acc) else ''}")

        # ── Checkpoint ────────────────────────────────────────────────────────
        torch.save({
            "epoch": epoch, "model": model.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "history": history,
            "best_val_acc": best_val_acc, "patience_cnt": patience_cnt,
        }, str(cfg["latest_ckpt_path"]))

        # ── Early stopping ────────────────────────────────────────────────────
        if patience_cnt >= cfg["early_stop_patience"]:
            print(f"\n⏹  Early stopping at epoch {epoch} (no improvement for {cfg['early_stop_patience']} checks)")
            break

    # ── SWA batch norm update ─────────────────────────────────────────────────
    print("\nUpdating SWA BatchNorm statistics...")
    bn_ld = DataLoader(
        DTMPointCloudDataset(cfg=cfg, split="train", augment=False),
        batch_size=cfg["batch_size"], num_workers=cfg["num_workers"]
    )
    torch.optim.swa_utils.update_bn(bn_ld, swa_model, device=device)
    torch.save(swa_model.module.state_dict(), cfg["swa_model_save_path"])
    print(f"✅ SWA model saved: {cfg['swa_model_save_path']}")

    with open(cfg["history_path"], "w") as f:
        json.dump(history, f, indent=2)

    return history, best_val_acc


# ── Plot helper ───────────────────────────────────────────────────────────────

def plot_training(history, save_path):
    val_h   = [h for h in history if not np.isnan(h.get("va", float("nan")))]
    all_ep  = [h["ep"] for h in history]
    val_ep  = [h["ep"] for h in val_h]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(all_ep, [h["tl"] for h in history], label="Train")
    if val_h:
        axes[0].plot(val_ep, [h["vl"] for h in val_h], label="Val")
    axes[0].set(title="Focal Loss", xlabel="Epoch"); axes[0].legend()

    axes[1].plot(all_ep, [h["ta"] for h in history], label="Train")
    if val_h:
        axes[1].plot(val_ep, [h["va"] for h in val_h], label="Val")
    axes[1].axhline(0.95, ls="--", color="red", label="95% target", alpha=0.6)
    axes[1].set(title="Accuracy", xlabel="Epoch", ylim=(0.8, 1.0)); axes[1].legend()

    axes[2].plot(all_ep, [h.get("tf1", 0) for h in history], label="Train F1")
    if val_h:
        axes[2].plot(val_ep, [h.get("vf1", 0) for h in val_h], label="Val F1")
    axes[2].set(title="Ground-class F1", xlabel="Epoch"); axes[2].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    print(f"✅ Training curves saved: {save_path}")


# ── RUN TRAINING ─────────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  STARTING TRAINING")
print("═"*60)
t0 = time.time()
history, best_val_acc = train(CFG, model)
elapsed = (time.time() - t0) / 60

print(f"\n{'═'*60}")
print(f"  TRAINING COMPLETE  {elapsed:.1f} min")
print(f"  Best val accuracy : {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")
print(f"{'═'*60}")
plot_training(history, CFG["curves_path"])


In [ ]:
# ─── Cell 7: Threshold Optimisation & Final Evaluation ───────────────────────
#
# Why this matters:
#   The model outputs P(ground) in [0,1]. Default threshold 0.5 is rarely
#   optimal for imbalanced classes. We sweep 0.05–0.95 and pick the threshold
#   that maximises F1 on the validation set. This typically adds +0.5–1.5%
#   accuracy without any retraining.
# ─────────────────────────────────────────────────────────────────────────────

import json, torch, numpy as np, matplotlib.pyplot as plt, torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm


def find_optimal_threshold(model_path: str, swa_path: str, cfg: dict, device):
    """
    Evaluates both the best epoch model and the SWA model on val set,
    picks whichever achieves higher accuracy at its optimal threshold.
    """
    _, _, _, va_ld = build_loaders(cfg)

    results = {}
    for label, path in [("best_epoch", model_path), ("swa", swa_path)]:
        if not Path(path).exists():
            continue
        m = DTMPointNet2(in_feat=7).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()

        all_probs, all_labels = [], []
        with torch.no_grad():
            for pts, labs in tqdm(va_ld, desc=f"Inference [{label}]", leave=False):
                pts = pts.to(device)
                logits = m(pts)                              # (B,N,2)
                probs  = F.softmax(logits, dim=-1)[:, :, 1] # P(ground)
                all_probs.append(probs.view(-1).cpu().numpy())
                all_labels.append(labs.view(-1).numpy())

        probs_arr  = np.concatenate(all_probs)
        labels_arr = np.concatenate(all_labels)

        # Sweep thresholds
        thresholds = np.arange(0.05, 0.95, 0.01)
        best_thr, best_f1, best_acc = 0.5, 0.0, 0.0
        thr_metrics = []

        for thr in thresholds:
            preds = (probs_arr >= thr).astype(np.int64)
            tp = ((preds == 1) & (labels_arr == 1)).sum()
            fp = ((preds == 1) & (labels_arr == 0)).sum()
            fn = ((preds == 0) & (labels_arr == 1)).sum()
            p  = tp / (tp + fp + 1e-9)
            r  = tp / (tp + fn + 1e-9)
            f1 = 2 * p * r / (p + r + 1e-9)
            acc = (preds == labels_arr).mean()
            thr_metrics.append((thr, f1, acc))
            if f1 > best_f1:
                best_f1, best_thr, best_acc = f1, thr, acc

        results[label] = {
            "model_path"  : path,
            "threshold"   : float(best_thr),
            "val_f1"      : float(best_f1),
            "val_accuracy": float(best_acc),
            "thr_metrics" : thr_metrics,
        }
        print(f"  [{label}]  best threshold={best_thr:.2f}  F1={best_f1:.4f}  acc={best_acc:.4f}")

    # Pick the winner
    winner = max(results, key=lambda k: results[k]["val_accuracy"])
    best   = results[winner]
    print(f"\n  ✅ Selected: {winner} (acc={best['val_accuracy']:.4f})")

    # Plot threshold curve
    tms  = best["thr_metrics"]
    thrs = [t[0] for t in tms]
    f1s  = [t[1] for t in tms]
    accs = [t[2] for t in tms]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thrs, f1s,  label="F1",       color="#2196F3")
    ax.plot(thrs, accs, label="Accuracy", color="#4CAF50")
    ax.axvline(best["threshold"], ls="--", color="red", label=f"Optimal thr={best['threshold']:.2f}")
    ax.axhline(0.95, ls=":", color="orange", alpha=0.7, label="95% target")
    ax.set(title="Threshold vs Accuracy/F1 (val set)",
           xlabel="Threshold", ylabel="Score", ylim=(0.8, 1.0))
    ax.legend(); plt.tight_layout()
    plt.savefig(str(Path(cfg["logs_dir"]) / "threshold_curve.png"), dpi=120)
    plt.show()

    # Save
    save_data = {
        "model"           : winner,
        "model_path"      : best["model_path"],
        "threshold"       : best["threshold"],
        "val_accuracy"    : best["val_accuracy"],
        "val_f1"          : best["val_f1"],
    }
    with open(cfg["threshold_path"], "w") as f:
        json.dump(save_data, f, indent=2)
    print(f"\n  Saved: {cfg['threshold_path']}")
    return save_data


result = find_optimal_threshold(
    CFG["model_save_path"],
    CFG["swa_model_save_path"],
    CFG, DEVICE
)

print(f"\n{'═'*60}")
print(f"  FINAL RESULT")
print(f"  Model         : {result['model']}")
print(f"  Threshold     : {result['threshold']:.2f}")
print(f"  Val Accuracy  : {result['val_accuracy']*100:.2f}%")
print(f"  Val F1        : {result['val_f1']:.4f}")
target_hit = result['val_accuracy'] >= 0.95
print(f"  Target (>95%) : {'✅ HIT' if target_hit else '⚠️  NOT YET — see tips below'}")
print(f"{'═'*60}")

if not target_hit:
    print("""
  TIPS TO REACH >95%:
  ━━━━━━━━━━━━━━━━━━
  1. Increase epochs: CFG['epochs'] = 100 and re-run Cell 6
  2. Increase tiles: ensure training_data.zip has all 18,373 tiles
  3. Lower focal_gamma to 1.5 if ground recall is already high
  4. Try focal_alpha = 0.80 if ground is still being missed
  5. Increase max_points_per_tile to 8192 (uses more VRAM)
""")


In [ ]:
# ─── Cell 8: Collect & Download Outputs ──────────────────────────────────────
from pathlib import Path
import zipfile, json

# Read threshold metadata to know which model file is the winner
with open(CFG["threshold_path"]) as f:
    meta = json.load(f)

WINNER_MODEL = meta["model_path"]   # best_model.pth or swa_model.pth

output_files = [
    WINNER_MODEL,
    CFG["swa_model_save_path"],
    CFG["model_save_path"],
    CFG["threshold_path"],
    CFG["history_path"],
    CFG["curves_path"],
    str(Path(CFG["logs_dir"]) / "threshold_curve.png"),
]

# Zip everything for one-click download
out_zip = Path("/kaggle/working/dtm_outputs.zip")
with zipfile.ZipFile(str(out_zip), "w", zipfile.ZIP_DEFLATED) as zf:
    for p in output_files:
        if Path(p).exists():
            zf.write(p, Path(p).name)
            print(f"  ✅ {Path(p).name}  ({Path(p).stat().st_size/1e3:.0f} KB)")
        else:
            print(f"  ⚠️  Not found: {p}")

print(f"\n📦 dtm_outputs.zip  ({out_zip.stat().st_size/1e6:.1f} MB)")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  NEXT STEPS — DOWNLOAD AND RETURN TO LOCAL MACHINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1. In the Kaggle sidebar → Output → dtm_outputs.zip → ⬇ Download
  2. Extract on your Ubuntu laptop
  3. Copy to:
       best_model.pth   →  .../GSI 2026/Logs/best_model.pth
       optimal_threshold.json → .../GSI 2026/Logs/
  4. Open dtm_generation_pipeline.ipynb in VS Code
  5. Run Section 8 (Inference) → Section 9 (DTM) → Section 10 (Hydrology)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
